In [24]:
import pandas as pd


In [25]:
df=pd.read_csv("Dataset.csv")
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [26]:
df1=df.copy(deep=True)

In [27]:
df1.describe()
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

Review rating has null values

In [28]:
df1['Review Rating'].isnull().sum()

np.int64(37)

Since Review Rating is numerical and customer satisfaction related:
Customers buying Footwear may rate differently from Clothing customers.

Category-wise median is more meaningful than overall mean.

In [29]:
df1['Review Rating'] = df1['Review Rating'].fillna(
    df.groupby('Category')['Review Rating'].transform('median')
)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,No,32,Venmo,Weekly
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,No,41,Bank Transfer,Bi-Weekly
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,No,24,Venmo,Quarterly
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,No,24,Venmo,Weekly


In [7]:
df1.drop_duplicates(inplace=True)

Standarized text columns

In [8]:
text_cols = df1.select_dtypes('object').columns

for col in text_cols:
    df1[col] = df1[col].str.strip()

Converting binary columns to 1 and 0 

In [9]:
binary_cols = [
    'Subscription Status',
    'Discount Applied',
    'Promo Code Used'
]

for col in binary_cols:
    df1[col] = df1[col].map({'Yes':1,'No':0})

Feature Engineering

Feature 1: Purchase Frequency Score

What it captures :-

This metric estimates how often a customer interacts with the brand over a year.

Why it matters :-

A customer purchasing weekly contributes more opportunities for revenue than one purchasing quarterly.

Business Question Answered :-

Which customers are most likely to remain active in the future?

In [10]:
freq_map = {
    'Weekly':52,
    'Bi-Weekly':26,
    'Fortnightly':26,
    'Monthly':12,
    'Quarterly':4,
    'Every 3 Months':4,
    'Annually':1
}

df1['Purchase_Freq_Score'] = df1['Frequency of Purchases'].map(freq_map)

Feature 2: Estimated Annual Spend

Metric:-

Annual Spend = Purchase Amount × Purchase Frequency

What it captures:-

Expected yearly revenue contribution.

Why it matters:-

Provides a more realistic view of customer value than transaction amount alone.

Business Question Answered:-

Who are the highest-value customers?


In [11]:
df1['Estimated_Annual_Spend'] = (
    df1['Purchase Amount (USD)']
    * df1['Purchase_Freq_Score']
)

Feature 3: Loyalty Score

Metric:-

Combination of:

Previous Purchases,
Purchase Frequency,
Subscription Status

What it captures:-

Overall customer commitment to the brand.

Why it matters:-

Customers with high loyalty scores are more likely to generate future revenue and require lower acquisition costs.

Business Question Answered:-

Who are genuinely loyal customers?

In [12]:
df1['Loyalty_Score'] = (
      0.4*df1['Previous Purchases']
    + 0.4*df1['Purchase_Freq_Score']
    + 0.2*df1['Subscription Status']
)

Feature 4: Promo Dependency Score

Metric Uses:-

Discount Applied,
Promo Code Used

What it captures:-

Degree to which a customer's purchasing behavior depends on incentives.

Interpretation:-
Score	Meaning
0	Rarely influenced by promotions , 
1	Moderately influenced , 
2	Highly promotion-dependent ,

Business Question Answered:-

Is the discount strategy creating loyalty or attracting bargain hunters?

In [13]:
df1['Promo_Dependency'] = (
      df1['Discount Applied']
    + df1['Promo Code Used']
)

Feature5: Value Tier

Purpose:-

Classify customers based on their revenue contribution.

What It Captures:-

Customer monetary value.

Business Question Answered:-

Who are the most valuable customers?

What characteristics define the ideal customer?

In [14]:
df1['Annual_Spend'] = (
    df1['Purchase Amount (USD)']
    * df1['Purchase_Freq_Score']
)

In [15]:
df1['Value_Tier'] = pd.qcut(
    df1['Annual_Spend'],
    q=4,
    labels=[
        'Low Value',
        'Medium Value',
        'High Value',
        'Premium Value'
    ]
)

Feature :6 Customer Value Index

Metric combines:

Annual Spend,
Review Rating

What it captures:-

Balance between monetary value and satisfaction.

Why it matters:-

Identifies customers who are both profitable and happy.

Business Question Answered:-

What characteristics define the most valuable customers?

In [16]:
df1['Customer_Value_Index'] = (
    df1['Estimated_Annual_Spend']
    * (df1['Review Rating']/5)
)

Feature 7: Satisfaction Segment

Metric:-

Low, Medium, High satisfaction categories.

What it captures:-

Customer sentiment toward the brand.

Why it matters:-

Highly satisfied customers are more likely to:

Repurchase,
Recommend the brand,
Remain loyal

Business Question Answered:-

How does satisfaction relate to retention and spending?

In [17]:
df1['Satisfaction_Level'] = pd.cut(
    df1['Review Rating'],
    bins=[0,3,4,5],
    labels=['Low','Medium','High']
)

Feature 8: Customer Tenure Proxy

Metric:-

Customers classified as:

New,
Growing,
Veteran

What it captures:-

Approximate relationship length with the brand.

Why it matters:-

Longer-tenure customers typically have higher lifetime value.

Business Question Answered:-

Which customers have developed long-term relationships with the brand?

In [18]:
df1['Tenure_Proxy'] = pd.cut(
    df1['Previous Purchases'],
    bins=[0,15,30,50],
    labels=['New','Growing','Veteran']
)

Feature 9: Bargain Hunter Flag

Metric:-

Flag customers who use both:

Discounts,
Promo Codes

What it captures:-

Strong promotional reliance.

Why it matters:-

These customers may stop purchasing if promotions are reduced.

Business Question Answered:-

Who buys because they love the brand versus because of discounts?

In [19]:
df1['Bargain_Hunter'] = (
    (df1['Discount Applied']==1)
    &
    (df1['Promo Code Used']==1)
).astype(int)

Feature 10: Ideal Customer Score

Metric Combines:-

Annual Spend,
Loyalty,
Satisfaction,
Purchase History

What it captures:-

Overall customer desirability from a business perspective.

Why it matters:-

Creates a measurable definition of the Ideal Customer Profile (ICP).

Business Question Answered:-

What does the brand's ideal customer actually look like?

In [20]:
df1['Ideal_Customer_Score'] = (
      0.35*df1['Estimated_Annual_Spend']
    + 0.30*df1['Review Rating']
    + 0.20*df1['Previous Purchases']
    + 0.15*df1['Subscription Status']
)

In [21]:
df1.info()
df1.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   Customer ID             3900 non-null   int64   
 1   Age                     3900 non-null   int64   
 2   Gender                  3900 non-null   object  
 3   Item Purchased          3900 non-null   object  
 4   Category                3900 non-null   object  
 5   Purchase Amount (USD)   3900 non-null   int64   
 6   Location                3900 non-null   object  
 7   Size                    3900 non-null   object  
 8   Color                   3900 non-null   object  
 9   Season                  3900 non-null   object  
 10  Review Rating           3900 non-null   float64 
 11  Subscription Status     3900 non-null   int64   
 12  Shipping Type           3900 non-null   object  
 13  Discount Applied        3900 non-null   int64   
 14  Promo Code Used         

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,...,Estimated_Annual_Spend,Loyalty_Score,Promo_Dependency,Annual_Spend,Value_Tier,Customer_Value_Index,Satisfaction_Level,Tenure_Proxy,Bargain_Hunter,Ideal_Customer_Score
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,1378,16.2,2,1378,High Value,854.36,Medium,New,1,486.18
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,1664,11.4,2,1664,Premium Value,1031.68,Medium,New,1,583.88
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,3796,30.2,2,3796,Premium Value,2353.52,Medium,Growing,1,1334.28
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,4680,40.6,2,4680,Premium Value,3276.00,Medium,Veteran,1,1649.00
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,49,13.0,2,49,Low Value,26.46,Low,Veteran,1,24.31


In [22]:
df1.to_csv("Featured_dataset.csv",index=False)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,...,Estimated_Annual_Spend,Loyalty_Score,Promo_Dependency,Annual_Spend,Value_Tier,Customer_Value_Index,Satisfaction_Level,Tenure_Proxy,Bargain_Hunter,Ideal_Customer_Score
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,1378,16.2,2,1378,High Value,854.36,Medium,New,1,486.18
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,1664,11.4,2,1664,Premium Value,1031.68,Medium,New,1,583.88
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,3796,30.2,2,3796,Premium Value,2353.52,Medium,Growing,1,1334.28
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,4680,40.6,2,4680,Premium Value,3276.00,Medium,Veteran,1,1649.00
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,49,13.0,2,49,Low Value,26.46,Low,Veteran,1,24.31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,...,1456,33.6,0,1456,High Value,1223.04,High,Veteran,0,517.26
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,...,1274,26.8,0,1274,High Value,1146.60,High,Veteran,0,455.45
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,...,132,11.2,0,132,Low Value,76.56,Low,Growing,0,51.87
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,...,4004,30.4,0,4004,Premium Value,3043.04,Medium,Growing,0,1407.34
